In [12]:
import sys
import os
sys.path.append(os.path.abspath('../..'))
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
from model_utils.plots import plot_results
from LSTM.lstm import LSTM
from GNN.GNNs.GCN_LSTM import Joint_GCN_LSTM
from GNN.GNNs.GCN import GCN
from dataset import TimeSeriesDataset
import time
import itertools
import math
import math

In [2]:
import os
import shutil
import stat
import time

def handle_remove_readonly(func, path, exc):
    # Callback to handle read-only files on Windows
    excvalue = exc[1]
    if func in (os.rmdir, os.remove, os.unlink) and excvalue.errno == 13: # EACCES
        os.chmod(path, stat.S_IWRITE)
        func(path)
    else:
        raise

# Clean up directories from previous runs
dirs_to_cleanup = ['best_models', 'grid_search_plots', 'training_logs', 'inference_logs']
for dir_path in dirs_to_cleanup:
    if os.path.exists(dir_path):
        # Retry a few times in case of transient locks
        for i in range(3):
            try:
                shutil.rmtree(dir_path, ignore_errors=False, onerror=handle_remove_readonly)
                print(f"Removed directory: {dir_path}")
                break
            except Exception as e:
                if i < 2:
                    time.sleep(1) # Wait a bit before retrying
                else:
                    print(f"Error removing {dir_path}: {e}")

Removed directory: grid_search_plots


In [3]:
import holidays

# -----------------------------------------------------------------------------
# DATA LOADING & PREPROCESSING
# -----------------------------------------------------------------------------
DATA_PATH = '../../dataset/data_andre.feather'  # Adjust path if needed
print(f"Loading data from {DATA_PATH}...")
# Use pd.read_feather instead of feather.read_table to get a DataFrame directly
df = pd.read_feather(DATA_PATH)

# Define constants
DATE_COL = 'date'
TARGET_COL = 'value'  # Assuming 'value' is the target variable (sales at the end of the day)


# Handle DATE_COL (ensure it is a column and not in the index)
if DATE_COL in df.index.names:
    if DATE_COL in df.columns:
        # If it's in both, drop the index version to avoid "cannot insert" error
        df = df.reset_index(drop=True)
    else:
        # If it's only in the index, move it to a column
        df = df.reset_index()

# If DATE_COL is NOT in columns (and wasn't in index), we have a problem, but assuming it exists somewhere.
# Just to be safe, if we still have a complex index, reset it.
if df.index.name == DATE_COL:
     df = df.reset_index(drop=True)
     
# Fallback: simple reset to ensure RangeIndex 0..N
df = df.reset_index(drop=True)

# Ensure date is datetime and sort
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(["store_id", "item_id", DATE_COL]).reset_index(drop=True)

# Prepare target variable
# Using 'value' as the target (sales at the end of the day)
y = df[TARGET_COL]
print(len(y))

# Calandar-based features
# Ensure datetime
df[DATE_COL] = pd.to_datetime(df[DATE_COL])

# Base calendar parts
df["day_of_week"]  = df[DATE_COL].dt.dayofweek.astype(int)
df["day_of_month"] = df[DATE_COL].dt.day.astype(int)         
df["moy"]          = df[DATE_COL].dt.month.astype(int)-1      
df["doy"]          = df[DATE_COL].dt.dayofyear.astype(int)-1   
df["is_weekend"] = (
    (df[DATE_COL].dt.dayofweek == 5) |
    (df[DATE_COL].dt.dayofweek == 6)
).astype(int)
# Create holiday calendar
df["time_idx"] = np.arange(len(df))
us_holidays = holidays.US()

# Holiday flag
df["is_holiday"] = df[DATE_COL].apply(lambda x: 1 if x in us_holidays else 0)

# Specific holidays
df["is_christmas"] = (df[DATE_COL].dt.month == 12) & (df[DATE_COL].dt.day == 25)

df["is_thanksgiving"] = df[DATE_COL].apply(
    lambda x: 1 if us_holidays.get(x) == "Thanksgiving Day" else 0
)

df["is_christmas"] = df["is_christmas"].astype(int)

# -----------------------------------------------------------------------------
# Add lag / rolling features
# -----------------------------------------------------------------------------
df["lag_1"] = df.groupby(["store_id", "item_id"])[TARGET_COL].shift(1)
df["lag_2"] = df.groupby(["store_id", "item_id"])[TARGET_COL].shift(2)
df["lag_3"] = df.groupby(["store_id", "item_id"])[TARGET_COL].shift(3)
df["lag_7"] = df.groupby(["store_id", "item_id"])[TARGET_COL].shift(7)
#df["lag_14"] = df.groupby(["store_id", "item_id"])[TARGET_COL].shift(14)
#df["lag_28"] = df.groupby(["store_id", "item_id"])[TARGET_COL].shift(28)

df["rolling_mean_3"] = df.groupby(["store_id", "item_id"])[TARGET_COL].transform(lambda x: x.shift(1).rolling(window=3).mean())
df["rolling_mean_7"] = df.groupby(["store_id", "item_id"])[TARGET_COL].transform(lambda x: x.shift(1).rolling(window=7).mean())

#LAG_COLS = ["lag_1", "lag_2", "lag_3", "lag_7"]
#ROLLING_COLS = ["rolling_mean_3", "rolling_mean_7"]
PROMO_BIN_COLS = [col for col in df.columns if col.startswith("promo_type_")]
PROMO_CONT_COLS = [col for col in df.columns if col.startswith("promo_value_")]
#df[LAG_COLS] = df[LAG_COLS].fillna(0)
#df[ROLLING_COLS] = df[ROLLING_COLS].fillna(0)

# Fill NaNs with 0 to keep the initial data rows instead of dropping them
df = df.fillna(0).reset_index(drop=True)
LAG_COLS = ["lag_1", "lag_2", "lag_3", "lag_7"]
#ROLLING_COLS = ["rolling_mean_3", "rolling_mean_7"]
CATEGORICAL_COLS = ["day_of_week", "doy"]
#CURSE_OF_DIMENSIONALITY_COLS = ["doy"]  # This column creates 365 OHE columns, which is problematic for LSTM input
CONTINUOUS_COLS = []
BINARY_COLS = [] 

#EXOG_COLS = CATEGORICAL_COLS + CONTINUOUS_COLS + BINARY_COLS
EXOG_COLS = CATEGORICAL_COLS # Exclude categorical for now to avoid OHE explosion
# Reorder columns to have date , target and exogenous variables first (optional, for better readability)
df = df[[DATE_COL, TARGET_COL] + [col for col in df.columns if col not in [DATE_COL, TARGET_COL] + EXOG_COLS] + EXOG_COLS]

print(PROMO_BIN_COLS)
print(PROMO_CONT_COLS)
# -----------------------------------------------------------------------------
# DATA SPLITTING
# -----------------------------------------------------------------------------
# Define split sizes
train_size = 455
val_size = 154
forecast_horizon = 152

# ensure day 2022-09-24 is the first day of test set
df = df.sort_values(["store_id", "item_id", DATE_COL]).reset_index(drop=True)

lookback_window = 7

Loading data from ../../dataset/data_andre.feather...
1082371
['promo_type_FRPG', 'promo_type_GAS', 'promo_type_BOGO', 'promo_type_DISC', 'promo_type_CIRC', 'promo_type_CIRE', 'promo_type_CLCP', 'promo_type_LFPE']
['promo_value_FRPG', 'promo_value_GAS', 'promo_value_BOGO', 'promo_value_DISC', 'promo_value_CIRC', 'promo_value_CIRE', 'promo_value_CLCP', 'promo_value_LFPE']


In [4]:
df

,date,value,item_id,cat_label,sdep_label,dep_label,dmn_label,promo_type_FRPG,promo_value_FRPG,promo_type_GAS,...,is_christmas,is_thanksgiving,lag_1,lag_2,lag_3,lag_7,rolling_mean_3,rolling_mean_7,day_of_week,doy
0,2021-01-23,8,27,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,0.0,0.0,0.0,0.0,0.000000,0.000000,5,22
1,2021-01-24,14,27,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,8.0,0.0,0.0,0.0,0.000000,0.000000,6,23
2,2021-01-25,8,27,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,14.0,8.0,0.0,0.0,0.000000,0.000000,0,24
3,2021-01-26,5,27,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,8.0,14.0,8.0,0.0,10.000000,0.000000,1,25
4,2021-01-27,8,27,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,5.0,8.0,14.0,0.0,9.000000,0.000000,2,26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1082366,2023-02-18,28,900087600,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,0,0.0,0,...,0,0,44.0,45.0,55.0,41.0,48.000000,43.428571,5,48
1082367,2023-02-19,46,900087600,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,0,0.0,0,...,0,0,28.0,44.0,45.0,31.0,39.000000,41.571429,6,49
1082368,2023-02-20,28,900087600,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,0,0.0,0,...,0,0,46.0,28.0,44.0,53.0,39.333333,43.714286,0,50
1082369,2023-02-21,31,900087600,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,0,0.0,0,...,0,0,28.0,46.0,28.0,35.0,34.000000,40.142857,1,51


In [5]:
print(df[TARGET_COL].describe())

count    1.082371e+06
mean     6.387899e+00
std      1.132080e+01
min      0.000000e+00
25%      1.000000e+00
50%      4.000000e+00
75%      8.000000e+00
max      1.000000e+03
Name: value, dtype: float64


# Graph Data Validation
Load and inspect graph data to confirm it is suitable for the Joint GCN-LSTM approach.

In [6]:
import torch
from torch_geometric.utils import to_scipy_sparse_matrix
import scipy.sparse as sp
import numpy as np

GRAPH_DATA_DIR = './graph_data'
GRAPH_FILE = 'CID_Graph_data.pt'   # Change to DTW_Graph_data.pt or StandardScaled_Distance_Graph_data.pt as needed

# ── 1. Load ────────────────────────────────────────────────────────────────────
pyg_data = torch.load(f"{GRAPH_DATA_DIR}/{GRAPH_FILE}", weights_only=False)
graph_name = GRAPH_FILE.replace(".pt", "")
print(f"=== {graph_name} ===\n")

# ── 2. Basic PyG object checks ────────────────────────────────────────────────
print(f"PyG Data object:\n{pyg_data}\n")

num_nodes = pyg_data.x.size(0)
num_node_features = pyg_data.x.size(1)
num_edges = pyg_data.edge_index.size(1)

print(f"Num nodes       : {num_nodes}")
print(f"Node feature dim: {num_node_features}  (= length of each item's time-series)")
print(f"Num edges       : {num_edges}")
print(f"Avg degree      : {num_edges / num_nodes:.2f}")

# ── 3. Node feature sanity checks ─────────────────────────────────────────────
x = pyg_data.x
print(f"\nNode features  — shape : {x.shape}")
print(f"  dtype  : {x.dtype}  (must be float32 for GCN)")
print(f"  NaNs   : {torch.isnan(x).sum().item()}")
print(f"  Infs   : {torch.isinf(x).sum().item()}")
print(f"  min    : {x.min().item():.4f}")
print(f"  max    : {x.max().item():.4f}")
print(f"  mean   : {x.mean().item():.4f}")

# ── 4. Edge index sanity checks ───────────────────────────────────────────────
edge_index = pyg_data.edge_index
print(f"\nEdge index — shape: {edge_index.shape}")
print(f"  dtype         : {edge_index.dtype}  (must be int64 for sparse ops)")
print(f"  min node idx  : {edge_index.min().item()}")
print(f"  max node idx  : {edge_index.max().item()}  (must be < num_nodes={num_nodes})")

# Check for self-loops
self_loops = (edge_index[0] == edge_index[1]).sum().item()
print(f"  Self-loops    : {self_loops}")

# Check for symmetry (undirected graph)
edge_set = set(map(tuple, edge_index.t().tolist()))
reverse_set = set((j, i) for i, j in edge_set)
symmetric = edge_set == reverse_set
print(f"  Symmetric     : {symmetric}  (undirected graph expected)")

# ── 5. Build Sparse Adjacency Matrix (same as used in GCN) ───────────────────
adj_scipy = to_scipy_sparse_matrix(edge_index, num_nodes=num_nodes)
adj_scipy = adj_scipy + adj_scipy.T.multiply(adj_scipy.T > adj_scipy) - adj_scipy.multiply(adj_scipy.T > adj_scipy)
adj_scipy = adj_scipy + sp.eye(adj_scipy.shape[0])  # Add self-loops
rowsum = np.array(adj_scipy.sum(1))
d_inv_sqrt = np.power(rowsum, -0.5).flatten()
d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.
d_mat_inv_sqrt = sp.diags(d_inv_sqrt)
adj_norm = adj_scipy.dot(d_mat_inv_sqrt).transpose().dot(d_mat_inv_sqrt)

print(f"\nNormalised adjacency matrix:")
print(f"  shape     : {adj_norm.shape}")
print(f"  nnz       : {adj_norm.nnz}   (non-zero entries incl. self-loops)")
print(f"  density   : {adj_norm.nnz / (num_nodes ** 2) * 100:.3f} %")

# ── 6. GCN forward-pass smoke test ───────────────────────────────────────────
from GNN.GNNs.GCN import GCN

NHID = 16
EMBED_DIM = 16
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

adj_coo = adj_norm.tocoo()
indices = torch.from_numpy(np.vstack((adj_coo.row, adj_coo.col)).astype(np.int64))
values  = torch.from_numpy(adj_coo.data.astype(np.float32))
adj_sparse = torch.sparse_coo_tensor(indices, values, torch.Size(adj_coo.shape)).to(DEVICE)

x_dev = x.to(DEVICE)
gcn_test = GCN(nfeat=num_node_features, nhid=NHID, nclass=EMBED_DIM, dropout=0.0).to(DEVICE)
gcn_test.eval()

with torch.no_grad():
    embeddings = gcn_test(x_dev, adj_sparse)

print(f"\nGCN smoke-test passed ✓")
print(f"  Input  shape : {x_dev.shape}")
print(f"  Output shape : {embeddings.shape}  (num_nodes × embed_dim)")
print(f"  NaNs in output: {torch.isnan(embeddings).sum().item()}")

# ── 7. Node-to-item_id mapping check ─────────────────────────────────────────
if hasattr(pyg_data, 'node_ids'):
    print(f"\nNode-to-item_id mapping present: {pyg_data.node_ids[:10]} ...")
else:
    print("\n⚠  No 'node_ids' attribute found on pyg_data.")
    print("   To look up which GCN row corresponds to which item_id during training,")
    print("   you need a node_ids list. Check utils.nx_to_pyg_data — add it there if missing.")

print("\n=== Validation complete ===")


c:\Users\Andre Silva\Desktop\Dissertation25-26\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== CID_Graph_data ===

PyG Data object:
Data(x=[100, 761], edge_index=[2, 2], edge_attr=[2, 1], node_ids=[100])

Num nodes       : 100
Node feature dim: 761  (= length of each item's time-series)
Num edges       : 2
Avg degree      : 0.02

Node features  — shape : torch.Size([100, 761])
  dtype  : torch.float32  (must be float32 for GCN)
  NaNs   : 0
  Infs   : 0
  min    : -1.6904
  max    : 7.7745
  mean   : -0.0000

Edge index — shape: torch.Size([2, 2])
  dtype         : torch.int64  (must be int64 for sparse ops)
  min node idx  : 47
  max node idx  : 48  (must be < num_nodes=100)
  Self-loops    : 0
  Symmetric     : True  (undirected graph expected)

Normalised adjacency matrix:
  shape     : (100, 100)
  nnz       : 102   (non-zero entries incl. self-loops)
  density   : 1.020 %

GCN smoke-test passed ✓
  Input  shape : torch.Size([100, 761])
  Output shape : torch.Size([100, 16])  (num_nodes × embed_dim)
  NaNs in output: 0

Node-to-item_id mapping present: [676, 688, 691, 69

In [7]:
GCN_embed_dim=16


# Grid Search Cell

In [8]:
import time
import itertools
import sys
import os
import importlib
sys.path.append(os.path.abspath('..'))
import model_utils.plots
importlib.reload(model_utils.plots)


# Modify lstm_experiment to return timings and handle plotting internally
def joint_experiment_grid(df, target, item_id, store_id, train_size=500, val_size=100, forecast_window=161, 
                   seq_length=30, epochs=100, batch_size=32, lr=0.001, dropout=0.0, hidden_size=32, num_layers=1, exog_cols=None,
                   patience=50, seed=42, loss_type='MSELoss', save_plot_path=None, use_best_model=True):

    
    gcn_embed_dim = 16 
    base_gcn = GCN(nfeat=graph_x.shape[1], nhid=32, nclass=gcn_embed_dim, dropout=0.5).to(device)
    # Set seed for reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    
    test_start_idx = len(df) - forecast_window
    val_start_idx = test_start_idx - val_size
    train_start_idx = val_start_idx - train_size

    print(f"Calculated indices - Train Start: {train_start_idx}, Val Start: {val_start_idx}, Test Start: {test_start_idx}")
    # Safety check
    if train_start_idx < 0:
        # If dataset is too short, we just start from 0
        train_start_idx = 0
        print(f"Warning: Dataset shorter than requested split sizes. adjusting train_start to 0.")

    train_slice = slice(train_start_idx, val_start_idx)
    val_slice = slice(val_start_idx, test_start_idx)
    test_slice = slice(test_start_idx, None)
    
    # Extract Target
    train = df[target][train_slice].values
    val = df[target][val_slice].values
    test = df[target][test_slice].values
    
    # Scale Target
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train.reshape(-1, 1)).flatten()
    val_scaled = scaler.transform(val.reshape(-1, 1)).flatten()

    # Extract Exogenous Variables
    exog_train = None
    exog_val = None
    full_exog = None
    
    if exog_cols:
        exog_train = df[exog_cols][train_slice].values
        exog_val = df[exog_cols][val_slice].values
        exog_test = df[exog_cols][test_slice].values
        # Scale Exogenous Variables
        exog_scaler = MinMaxScaler()
        exog_train_scaled = exog_scaler.fit_transform(exog_train)
        exog_val_scaled = exog_scaler.transform(exog_val)
        exog_test_scaled = exog_scaler.transform(exog_test)
        
    # Input size = 1 (target) + number of exog features
    input_size = 1 + (len(exog_cols) if exog_cols else 0)
    
    # -------------------------------------------------------------------------
    # 2. Datasets & Loaders
    # -------------------------------------------------------------------------
    # Pass exogenous data to TimeSeriesDataset
    train_dataset = TimeSeriesDataset(train_scaled, exog_train_scaled, seq_length)
    use_pin_memory = torch.cuda.is_available()
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, pin_memory=use_pin_memory)
    
    # Important: Pass exog_val to validation dataset too!
    val_dataset = TimeSeriesDataset(val_scaled, exog_val_scaled, seq_length)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, pin_memory=use_pin_memory)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, dropout=dropout).to(device)
    
    if loss_type == 'MSELoss':
        criterion = nn.MSELoss()
    elif loss_type == 'L1Loss':
        criterion = nn.L1Loss()
    elif loss_type == 'HuberLoss':
        criterion = nn.HuberLoss()
    else:
        raise ValueError(f"Unsupported loss_type: {loss_type}")

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion2 = nn.MSELoss()  # For validation loss calculation
    
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    
    # Directories
    model_dir = f'best_models/seed_{seed}/{loss_type}'
    os.makedirs(model_dir, exist_ok=True)
    best_model_path = f'{model_dir}/lstm_item{item_id}_store{store_id}.pth'
    
    log_dir = f'training_logs/seed_{seed}/{loss_type}'
    os.makedirs(log_dir, exist_ok=True)
    log_path = f'{log_dir}/lstm_item{item_id}_store{store_id}_log.csv'

    best_epoch = 0
    
    # -------------------------------------------------------------------------
    # 3. Training Loop
    # -------------------------------------------------------------------------
    start_train_time = time.time()
    
    with open(log_path, 'w') as log_file:
        log_file.write("Epoch,Batch,Batch_X,Batch_Y,Output,Loss\n")
        
        for epoch in range(epochs):
            model.train()
            epoch_loss = 0
            for batch_idx, (batch_x, batch_y) in enumerate(train_loader):
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                optimizer.zero_grad()
                outputs = model(batch_x,graph_x, graph_adj, target_node_idx)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item()
                
                # Convert tensors to lists (represented as strings) to log in CSV
                batch_x_str = str(batch_x.cpu().detach().numpy().tolist()).replace('"', "'")
                batch_y_str = str(batch_y.cpu().detach().numpy().tolist()).replace('"', "'")
                outputs_str = str(outputs.cpu().detach().numpy().tolist()).replace('"', "'")
                
                log_file.write(f'{epoch},{batch_idx},"{batch_x_str}","{batch_y_str}","{outputs_str}",{loss.item()}\n')
            
            avg_train_loss = epoch_loss / len(train_loader)
            train_losses.append(avg_train_loss)
            
            # Validation
            model.eval()
            all_outputs = []
            all_targets = []

            with torch.no_grad():
                for batch_idx, (batch_x, batch_y) in enumerate(val_loader):
                    batch_x = batch_x.to(device)
                    batch_y = batch_y.to(device)
                    
                    outputs = model(batch_x,graph_x, graph_adj, target_node_idx)
                    all_outputs.append(outputs)
                    all_targets.append(batch_y)

            all_outputs = torch.cat(all_outputs, dim=0).view(-1)
            all_targets = torch.cat(all_targets, dim=0).view(-1)
            
            val_loss = criterion2(all_outputs, all_targets).item()
            val_losses.append(val_loss)
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_epoch = epoch + 1
                if use_best_model:
                    torch.save(model.state_dict(), best_model_path)
            
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.6f} | Val Loss: {val_loss:.6f}")

    if not use_best_model:
        torch.save(model.state_dict(), best_model_path)
        best_epoch = epochs
        print(f"Saved last model (Epoch {epochs}) to {best_model_path}")
    else:
        print(f"Saved best model (Epoch {best_epoch}) to {best_model_path}")

    train_time = time.time() - start_train_time

    # -------------------------------------------------------------------------
    # 4. Inference (Recursive)
    # -------------------------------------------------------------------------
    start_inference_time = time.time()
    
    if os.path.exists(best_model_path):
        model.load_state_dict(torch.load(best_model_path))
        if use_best_model:
            print (f"Loaded model from: {best_model_path} (Best Epoch: {best_epoch})")
        else:
            print (f"Loaded model from: {best_model_path} (Last Epoch)")
    else:
        print("Warning: No model saved. Using current model in memory.")

    model.eval()
    
    # Get the start date of the test set
    test_start_date = df[DATE_COL].iloc[test_start_idx]
    print(f"Test set starts on: {test_start_date.date()}")

    # Use the last seq_length points from validation data as initial input
    current_seq = val_scaled[-seq_length:].tolist()
    if exog_cols is not None:
        # Shift exog by 1 to match training dataloader
        current_exog_seq = exog_val_scaled[-seq_length + 1:].tolist() + [exog_test_scaled[0].tolist()]
        
    current_date_seq = df[DATE_COL].iloc[test_start_idx - seq_length : test_start_idx].dt.date.tolist()
    
    forecast = []
    
    # Setup inference log file
    inf_log_dir = f'inference_logs/seed_{seed}/{loss_type}'
    os.makedirs(inf_log_dir, exist_ok=True)
    inf_log_path = f'{inf_log_dir}/inference_item{item_id}_store{store_id}.csv'
    
    with open(inf_log_path, 'w') as inf_log_file:
        inf_log_file.write("Step,X,Predicted_Y\n")
        
        with torch.no_grad():
            for step in range(forecast_window):
                # Prepare input
                if exog_cols is not None:
                    # Combine target sequence with exog
                    current_seq_arr = np.array(current_seq).reshape(-1, 1)
                    current_exog_arr = np.array(current_exog_seq)
                    x_np = np.column_stack([current_seq_arr, current_exog_arr])
                else:
                    x_np = np.array(current_seq).reshape(-1, 1)
                
                x = torch.FloatTensor(x_np).unsqueeze(0).to(device)
                
                # Predict next value
                pred = model(x).cpu().numpy()[0, 0]
                forecast.append(pred)
                
                # Format X to log
                x_str = str(x_np.tolist()).replace('"', "'")
                inf_log_file.write(f'{step},"{x_str}",{pred}\n')
                
                # Print date info
                y_date = df[DATE_COL].iloc[test_start_idx + step].date()
                print(f"Step {step}: Predicting for Date: {y_date}")
                print(f"X element dates (len {len(current_date_seq)}): {current_date_seq}")
                
                # Update sequence (sliding window)
                current_seq = current_seq[1:] + [pred]
                current_date_seq = current_date_seq[1:] + [y_date]
                if exog_cols is not None:
                    # Update exog for the NEXT step using the future test set value
                    if step + 1 < forecast_window:
                        exog_future = exog_test_scaled[step + 1]
                        current_exog_seq = current_exog_seq[1:] + [exog_future.tolist()]
    
    # Inverse transform predictions
    forecast = scaler.inverse_transform(np.array(forecast).reshape(-1, 1)).flatten()
    print(f"Forecasted values {forecast[:]}")
    inference_time = time.time() - start_inference_time
    
    # Metrics
    rmse = np.sqrt(mean_squared_error(test, forecast))
    mae = mean_absolute_error(test, forecast)
    bias = np.mean(forecast - test)
    score = 0.5 * rmse + 0.25 * mae + 0.25 * abs(bias) 
    if save_plot_path:
        train_index = df[DATE_COL][train_slice].values
        val_index = df[DATE_COL][val_slice].values
        test_index = df[DATE_COL][test_slice].values
        
        plot_results(train, val, test, forecast, train_index, val_index, test_index, 
                     train_losses, val_losses, target, 
                     title=f'LSTM Forecast (Seed={seed}, Loss={loss_type}, Item={item_id}, Store={store_id})',
                     save_path=save_plot_path,
                     rmse=rmse, mae=mae, bias=bias, score=score)
    
    return rmse, mae, train_time, inference_time, best_epoch

In [9]:
# Grid Search Parameters
seeds = [2024]
loss_functions = ['MSELoss']  
batch_size = 32
hidden_size = 32
dropout = 0.0
EPOCHS = 150 # Ensure EPOCHS is defined
LEARNING_RATE = 0.001

# Filter for specific products if needed
target_products = [916110]
if target_products:
    products = df[df['item_id'].isin(target_products)][['item_id', 'store_id']].drop_duplicates().values
else:
    # Get all unique products from the subset dataset
    products = df[['item_id', 'store_id']].drop_duplicates().values

# Prepare results storage
results = []
os.makedirs('grid_search_plots', exist_ok=True)


In [10]:
# check products series count and check for any missing dates
item_id, store_id = products[0]  # Just checking the first product for now
# Filter data for the specific product
df_product = df[(df['item_id'] == item_id) & (df['store_id'] == store_id)].copy()
df_product

,date,value,item_id,cat_label,sdep_label,dep_label,dmn_label,promo_type_FRPG,promo_value_FRPG,promo_type_GAS,...,is_christmas,is_thanksgiving,lag_1,lag_2,lag_3,lag_7,rolling_mean_3,rolling_mean_7,day_of_week,doy
1028440,2021-01-23,60,916110,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,0.0,0.0,0.0,0.0,0.000000,0.000000,5,22
1028441,2021-01-24,29,916110,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,60.0,0.0,0.0,0.0,0.000000,0.000000,6,23
1028442,2021-01-25,5,916110,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,29.0,60.0,0.0,0.0,0.000000,0.000000,0,24
1028443,2021-01-26,13,916110,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,5.0,29.0,60.0,0.0,31.333333,0.000000,1,25
1028444,2021-01-27,31,916110,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,13.0,5.0,29.0,0.0,15.666667,0.000000,2,26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1029196,2023-02-18,41,916110,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,35.0,20.0,29.0,61.0,28.000000,31.571429,5,48
1029197,2023-02-19,35,916110,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,41.0,35.0,20.0,35.0,32.000000,28.714286,6,49
1029198,2023-02-20,23,916110,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,35.0,41.0,35.0,23.0,37.000000,28.714286,0,50
1029199,2023-02-21,25,916110,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,23.0,35.0,41.0,18.0,33.000000,28.714286,1,51


In [11]:


# Prepare results storage
results = []
os.makedirs('grid_search_plots', exist_ok=True)

# Run Grid Search
print(f"Starting Grid Search with {len(seeds)} seeds, {len(loss_functions)} loss functions and {len(products)} products...")

for seed in seeds:
    print(f"\n--- Processing Seed: {seed} ---")
    for loss_type in loss_functions:
        print(f"\n--- Processing Loss Type: {loss_type} ---")
        
        for item_id, store_id in products:
            print(f"Running: Seed={seed}, Loss={loss_type}, Item={item_id}, Store={store_id}")
            
            # Filter data for the specific product
            df_product = df[(df['item_id'] == item_id) & (df['store_id'] == store_id)].copy()
            
            # Handle DATE_COL (ensure it is a column and not in the index)
            if DATE_COL in df_product.index.names:
                if DATE_COL in df_product.columns:
                    # If it's in both, drop the index version to avoid "cannot insert" error
                    df_product = df_product.reset_index(drop=True)
                else:
                    # If it's only in the index, move it to a column
                    df_product = df_product.reset_index()

            # Fallback: simple reset to ensure RangeIndex 0..N
            df_product = df_product.reset_index(drop=True)

            df_product[DATE_COL] = pd.to_datetime(df_product[DATE_COL])
            df_product = df_product.sort_values(DATE_COL)
            df_product = df_product.reset_index(drop=True) # Final clean reset
            
            # Create directory for plots if it doesn't exist
            plot_dir = f'grid_search_plots/seed_{seed}/{loss_type}'
            os.makedirs(plot_dir, exist_ok=True)
            plot_filename = f'{plot_dir}/lstm_item{item_id}_store{store_id}.png'
            rmse, mae, train_time, infer_time, best_epoch = joint_experiment_grid(
                df=df_product, 
                target=TARGET_COL, 
                item_id=item_id,
                store_id=store_id,
                train_size=train_size, 
                val_size=val_size,
                forecast_window=forecast_horizon, 
                seq_length=lookback_window,
                epochs=EPOCHS,  # Use the global EPOCHS setting (e.g., 1000)
                batch_size=batch_size, 
                lr=LEARNING_RATE,
                dropout=dropout,
                hidden_size=hidden_size,
                num_layers=1,
                patience=100,
                seed=seed,
                loss_type=loss_type,
                save_plot_path=plot_filename
            )
            
            results.append({
                'seed': seed,
                'loss_type': loss_type,
                'item_id': item_id,
                'store_id': store_id,
                'batch_size': batch_size,
                'hidden_size': hidden_size,
                'dropout': dropout,
                'rmse': rmse,
                'mae': mae,
                'train_time': train_time,
                'inference_time': infer_time,
                'best_epoch': best_epoch,
                'plot_path': plot_filename
            })
        

# Convert to DataFrame
results_df = pd.DataFrame(results)
results_df.to_csv('grid_search_results.csv', index=False)

Starting Grid Search with 1 seeds, 1 loss functions and 1 products...

--- Processing Seed: 2024 ---

--- Processing Loss Type: MSELoss ---
Running: Seed=2024, Loss=MSELoss, Item=916110, Store=6269


NameError: name 'graph_x' is not defined